# 1. Import libaries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 2. Rescaling

In [2]:
# Load rfm data
raw_rfm = pd.read_parquet("data/staging/the_complete_journey_rfm_no_outlier.parquet", engine= "pyarrow")
raw_rfm.head()

,household_key,recency,frequency,monetary
0,1,6,86,4330.16
1,3,9,46,2519.75
2,5,9,39,757.07
3,6,5,242,5613.98
4,7,3,59,3350.77


In [3]:
# Clone the RFM values to a new dataframe
rfm_df = raw_rfm[[ 'recency', 'frequency', 'monetary']].copy()

# Instantiate
scaler = StandardScaler()

# Fit_transform
rfm_df_scaled = scaler.fit_transform(rfm_df)
rfm_df_scaled

array([[-0.38288671, -0.05261494,  1.01279346],
       [-0.06642417, -0.71831957,  0.02102064],
       [-0.06642417, -0.83481788, -0.94460489],
       ...,
       [-0.80483676,  1.32872218,  0.02764374],
       [-0.69934924, -0.13582802,  0.25402351],
       [-0.59386173,  0.18038169,  1.08492449]], shape=(1920, 3))

In [4]:
# Convert to dataframe
rfm_df_scaled = pd.DataFrame(rfm_df_scaled)
rfm_df_scaled.columns = ['recency', 'frequency', 'monetary']
rfm_df_scaled.head()

,recency,frequency,monetary
0,-0.382887,-0.052615,1.012793
1,-0.066424,-0.718320,0.021021
2,-0.066424,-0.834818,-0.944605
3,-0.488374,2.543633,1.716091
4,-0.699349,-0.501966,0.476267


## 3. Hopkin Statistics

In [5]:
# snipset of the Hopkins statistic function to find if the dataset is suitable for clustering or not
from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform
import numpy as np
from math import isnan

def hopkins(X):
    d = X.shape[1]
    #d = len(vars) # columns
    n = len(X) # rows
    m = int(0.1 * n) 
    nbrs = NearestNeighbors(n_neighbors=1).fit(X.values)
 
    rand_X = sample(range(0, n, 1), m)
 
    ujd = []
    wjd = []
    for j in range(0, m):
        u_dist, _ = nbrs.kneighbors(uniform(np.amin(X,axis=0),np.amax(X,axis=0),d).reshape(1, -1), 2, return_distance=True)
        ujd.append(u_dist[0][1])
        w_dist, _ = nbrs.kneighbors(X.iloc[rand_X[j]].values.reshape(1, -1), 2, return_distance=True)
        wjd.append(w_dist[0][1])
 
    H = sum(ujd) / (sum(ujd) + sum(wjd))
    if isnan(H):
        print(ujd, wjd)
        H = 0
 
    return H

In [6]:
# Pass a dataframe to the Hopkins statistic
hopkins(rfm_df_scaled)

np.float64(0.8600495517376399)

> Datasets show strong cluster tendency